# TinyStories — Generation Playground

In [6]:
import torch
import pickle
import os
import glob
from cs336_basics.bpe import train_bpe
from cs336_basics.tokenizer import Tokenizer
from cs336_basics.model import TransformerLM

DEVICE = 'mps' if torch.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: mps


## 1. Load tokenizer (trains BPE once, then caches)

In [7]:
TOKENIZER_CACHE = 'data/tokenizer_cache.pkl'
VOCAB_SIZE = 10000
SPECIAL_TOKENS = ['<|endoftext|>']

if os.path.exists(TOKENIZER_CACHE):
    print('Loading tokenizer from cache...')
    with open(TOKENIZER_CACHE, 'rb') as f:
        vocab, merges = pickle.load(f)
else:
    print('Training BPE (this takes a few minutes)...')
    vocab, merges = train_bpe('data/TinyStoriesV2-GPT4-train.txt', VOCAB_SIZE, SPECIAL_TOKENS)
    with open(TOKENIZER_CACHE, 'wb') as f:
        pickle.dump((vocab, merges), f)
    print('Saved to cache.')

tokenizer = Tokenizer(vocab, merges, SPECIAL_TOKENS)
EOT_ID = tokenizer.encode(SPECIAL_TOKENS[0])[0]
print(f'Vocab size: {len(vocab)}, EOT token id: {EOT_ID}')

Loading tokenizer from cache...
Vocab size: 10000, EOT token id: 256


## 2. Load model checkpoint

In [8]:
# Auto-picks latest checkpoint — change path to use a specific one
checkpoints = sorted(glob.glob('checkpoints/ckpt_step_*.pt'))
CHECKPOINT = checkpoints[-1]
print(f'Loading: {CHECKPOINT}')

model = TransformerLM(
    vocab_size=10000,
    context_length=256,
    num_layers=4,
    d_model=512,
    num_heads=16,
    d_ff=1344,
    device=DEVICE,
)

ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded step {ckpt["iteration"]} | params: {sum(p.numel() for p in model.parameters()):,}')

Loading: checkpoints/ckpt_step_4000.pt
Loaded step 4000 | params: 22,696,448


## 3. Generate

In [9]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 200, temperature: float = 0.8, top_p: float = 0.9) -> str:
    ids = tokenizer.encode(prompt)
    tokens = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

    for _ in range(max_new_tokens):
        ctx = tokens[:, -256:]
        positions = torch.arange(ctx.shape[1], device=DEVICE).unsqueeze(0)
        logits = model(ctx, positions)[:, -1, :]

        logits = logits / temperature
        probs = torch.softmax(logits, dim=-1)

        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumprobs = sorted_probs.cumsum(dim=-1)
        mask = cumprobs - sorted_probs > top_p
        sorted_probs[mask] = 0
        sorted_probs /= sorted_probs.sum()
        next_id = sorted_idx[0, torch.multinomial(sorted_probs[0], 1)]  # shape (1,)

        if next_id.item() == EOT_ID:
            break
        tokens = torch.cat([tokens, next_id.unsqueeze(0)], dim=1)

    return tokenizer.decode(tokens[0].tolist())

In [10]:
prompt = 'Once upon a time there was a little girl named Lily'
print(generate(prompt, max_new_tokens=200, temperature=0.8, top_p=0.9))

Once upon a time there was a little girl named Lily. Lily loved to play outside in the sun. One day, she saw a shiny motorcycle on the ground. Lily was very happy and wanted to show her friends.
Lily ran to the motorcycle and started to dig with it. She made sure she could find out what it was. As she dug, she found a big box in the middle of the motorcycle. She opened the box and found a shiny copper toy inside.
Lily was so happy! She showed the motorcycle to her friends. They all played together and had a great time. From that day on, Lily and her friends had lots of fun and playing together in the park.



In [11]:
# Try different prompts / settings
prompt = 'Tom and his dog'
print(generate(prompt, max_new_tokens=150, temperature=1.0, top_p=0.95))

Tom and his dog wanted the old chair. Tom was curious and asked the big bird, "Can you help me get the chair?" The big bird said, "No, I will help you."
Tom and the big dog left the old judge. They all worked together to get the chair. They found the big red mug and put it in a big pile. The big bird put the big surprise in its beak and flew up into the sky. Tom and the big dog looked for their work! They were so happy to be with the big, guard. Tom and the big dog became best friends and troubled together every day.

